In [1]:
import keras
from keras import Sequential
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from keras.callbacks import EarlyStopping
from keras.layers import(
    Dense, Conv2D, MaxPooling2D, 
    Input, RandomFlip, RandomRotation,
    RandomZoom, RandomTranslation, Flatten,
    GlobalAveragePooling2D, Dropout
    ) 
from tensorflow.keras.applications.resnet import preprocess_input
from keras.applications import ResNet50
from keras.models import Model
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

2026-06-20 11:42:35.851113: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-20 11:42:35.896130: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-20 11:42:37.334025: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
tf.config.list_physical_devices("GPU")

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [3]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()

In [4]:
y_train = y_train.squeeze()
y_test = y_test.squeeze()

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

In [5]:
# Definição de Parâmetros
NUM_CLASSES = 10
BATCH_SIZE = 128
EPOCHS_HEAD = 15
IMG_SIZE = (96,96)

In [6]:
# Separação do conjunto de validação
VAL_SIZE = 5000

X_val = X_train[:VAL_SIZE]
y_val = y_train[:VAL_SIZE]

X_tr= X_train[VAL_SIZE:]
y_tr = y_train[VAL_SIZE:]

In [7]:
# Data Augmentation
dat_augmentation = Sequential([
    RandomFlip("horizontal"), # Espelhamento
    RandomTranslation(0.03,0.03), # Desloca até 10% na horizontal e 10% na vertical
    RandomRotation(0.03), # 0 a 8% de rotação
    RandomZoom(0.03), # Aplicar zom na faixa de 0 a 10%
], name='data_augmentation')

I0000 00:00:1781966562.103916   40409 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9702 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [8]:
# Pré-processamento
def preprocess_train(img, label):
    img = tf.cast(img, tf.float32)
    img = dat_augmentation(img,training=True)
    img = tf.image.resize(img,IMG_SIZE)
    img = preprocess_input(img)
    return img, label

def preprocess_eval(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.resize(img,IMG_SIZE)
    img = preprocess_input(img)
    return img, label

In [9]:
# Construir os dadasets (treino, validação e etest)

train_ds = tf.data.Dataset.from_tensor_slices((X_tr,y_tr))
val_ds = tf.data.Dataset.from_tensor_slices((X_val,y_val))
test_ds = tf.data.Dataset.from_tensor_slices((X_test,y_test))

train_ds = train_ds.shuffle(buffer_size=45000)
train_ds = train_ds.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

test_ds = test_ds.map(preprocess_eval, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

In [10]:
# Carregar

base_model = ResNet50(
    input_shape=(IMG_SIZE[0],IMG_SIZE[1],3),
    include_top=False
)

base_model.trainable=False

In [11]:
# Montar o modelo final (base congelada + nova cabeça)

inputs = Input(
    shape=(IMG_SIZE[0],IMG_SIZE[1],3)
)

x = base_model(inputs,training=False)
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation='relu')(x)
x = Dropout(0.3)(x)

outputs = Dense(NUM_CLASSES,activation='softmax')(x)

model = Model(inputs, outputs, name="TL_ResNet50_CIFAR10")

In [12]:
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

Model: "TL_ResNet50_CIFAR10"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, 96, 96, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 3, 3, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,851,274 (90.99 MB)

 Trainable params: 263,562 (1.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [13]:
early_stop = EarlyStopping(
    min_delta=0.001,
    patience=5,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=[early_stop]
)

Epoch 1/15


2026-06-20 11:42:53.458797: I external/local_xla/xla/service/service.cc:163] XLA service 0x797f2c004420 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-06-20 11:42:53.458837: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2026-06-20 11:42:53.728711: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-06-20 11:42:55.324694: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 92302


  1/352 ━━━━━━━━━━━━━━━━━━━━ 1:25:02 15s/step - accuracy: 0.1172 - loss: 3.6452

I0000 00:00:1781966582.475428   40522 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


351/352 ━━━━━━━━━━━━━━━━━━━━ 0s 246ms/step - accuracy: 0.6460 - loss: 1.0800

2026-06-20 11:44:30.876972: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5257', 4 bytes spill stores, 4 bytes spill loads

2026-06-20 11:44:31.195022: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_5520', 100 bytes spill stores, 100 bytes spill loads



352/352 ━━━━━━━━━━━━━━━━━━━━ 118s 294ms/step - accuracy: 0.6463 - loss: 1.0787 - val_accuracy: 0.8404 - val_loss: 0.4645
Epoch 2/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 93s 264ms/step - accuracy: 0.7762 - loss: 0.6517 - val_accuracy: 0.8536 - val_loss: 0.4228
Epoch 3/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 90s 256ms/step - accuracy: 0.7932 - loss: 0.5927 - val_accuracy: 0.8636 - val_loss: 0.3984
Epoch 4/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 89s 251ms/step - accuracy: 0.8067 - loss: 0.5505 - val_accuracy: 0.8644 - val_loss: 0.3875
Epoch 5/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 87s 247ms/step - accuracy: 0.8153 - loss: 0.5264 - val_accuracy: 0.8628 - val_loss: 0.3863
Epoch 6/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 84s 237ms/step - accuracy: 0.8210 - loss: 0.5048 - val_accuracy: 0.8702 - val_loss: 0.3788
Epoch 7/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 81s 231ms/step - accuracy: 0.8204 - loss: 0.5037 - val_accuracy: 0.8746 - val_loss: 0.3763
Epoch 8/15
352/352 ━━━━━━━━━━━━━━━━━━━━ 81s 229ms/step - accuracy: 0.8347 - loss: 0.4746 - va

In [14]:
preds_proba = model.predict(test_ds)
preds = np.argmax(preds_proba, axis=1)

79/79 ━━━━━━━━━━━━━━━━━━━━ 11s 105ms/step


In [15]:
print(classification_report(y_test,preds))

              precision    recall  f1-score   support

           0       0.87      0.87      0.87      1000
           1       0.91      0.92      0.91      1000
           2       0.90      0.83      0.86      1000
           3       0.80      0.69      0.74      1000
           4       0.84      0.82      0.83      1000
           5       0.77      0.85      0.81      1000
           6       0.87      0.92      0.89      1000
           7       0.86      0.89      0.87      1000
           8       0.92      0.92      0.92      1000
           9       0.88      0.91      0.90      1000

    accuracy                           0.86     10000
   macro avg       0.86      0.86      0.86     10000
weighted avg       0.86      0.86      0.86     10000



In [17]:
# Fine-tuning
base_model.trainable = True

for layer in base_model.layers[:140]:
    layer.trainable= False

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD + 10,
    initial_epoch=history.epoch[-1] + 1,
    callbacks=[early_stop]
)

Epoch 14/25


2026-06-20 12:16:21.706147: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'input_add_reduce_fusion_4', 8 bytes spill stores, 8 bytes spill loads



352/352 ━━━━━━━━━━━━━━━━━━━━ 92s 220ms/step - accuracy: 0.7947 - loss: 0.6138 - val_accuracy: 0.8834 - val_loss: 0.4137
Epoch 15/25
352/352 ━━━━━━━━━━━━━━━━━━━━ 74s 211ms/step - accuracy: 0.8836 - loss: 0.3468 - val_accuracy: 0.9002 - val_loss: 0.2947
Epoch 16/25
352/352 ━━━━━━━━━━━━━━━━━━━━ 83s 237ms/step - accuracy: 0.9085 - loss: 0.2734 - val_accuracy: 0.9048 - val_loss: 0.3890
Epoch 17/25
352/352 ━━━━━━━━━━━━━━━━━━━━ 87s 247ms/step - accuracy: 0.9256 - loss: 0.2116 - val_accuracy: 0.9164 - val_loss: 0.2672
Epoch 18/25
352/352 ━━━━━━━━━━━━━━━━━━━━ 92s 260ms/step - accuracy: 0.9428 - loss: 0.1715 - val_accuracy: 0.9190 - val_loss: 0.2617
Epoch 19/25
352/352 ━━━━━━━━━━━━━━━━━━━━ 91s 257ms/step - accuracy: 0.9495 - loss: 0.1481 - val_accuracy: 0.9146 - val_loss: 0.2775
Epoch 20/25
352/352 ━━━━━━━━━━━━━━━━━━━━ 89s 253ms/step - accuracy: 0.9579 - loss: 0.1281 - val_accuracy: 0.9160 - val_loss: 0.2663
Epoch 21/25
352/352 ━━━━━━━━━━━━━━━━━━━━ 89s 254ms/step - accuracy: 0.9599 - loss: 0.121

In [18]:
model.evaluate(test_ds)

79/79 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - accuracy: 0.8982 - loss: 0.3179


[0.31117457151412964, 0.902400016784668]